# Train Transolver on the SU2 hypersonic dataset

Kaggle orchestrator. Attach the `zeteixeira/su2-hypersonic-sphere-cone`
dataset as input (753-case version), set Accelerator to GPU T4 x2,
Internet on, Run All.

**The accelerator has to be set in the notebook editor.** The kernel API
exposes only a boolean `enable_gpu` and has no accelerator-type field, so a
CLI push draws whatever Kaggle assigns, which in practice has been a single
P100. That card is compute capability 6.0 and Kaggle's torch build ships no
sm_60 kernels, so training cannot run on it at all. `scripts/train.py`
checks the card before doing any work and aborts in seconds naming the
capability, so a wrong draw is cheap to catch, but only the editor can fix
it.

Two things run from this notebook. The ensemble cell trains a five-member
deep ensemble at `M = 32`: same split seed, members differing only in
`--init-seed` (0-4), every member trained from scratch. Set `TAG` to name
the run:

- `v2`, on the 727-case sweep alone, is the control
- `v3`, on the sweep plus the 26 active-learning loop cases, is the treatment

Both share `--seed 0`, so the core val/test/interpolation tiers are the same
cases in both; the loop cases carry `group_name='loop'` and route to train
only. The single difference between the two ensembles is the loop data,
which is what makes the before/after comparison meaningful.

The prior-ablation cell below trains the low-data cells instead. Run one or
the other, not both.

Outputs land in `/kaggle/working/run_*/` and survive the session as notebook
output.

**Run one seed per notebook copy, and never re-push a notebook whose output
you still need.** Kaggle keeps only the latest version's output, so pushing
a second run to the same notebook silently destroys the first run's
checkpoints. Give each seed its own notebook. Free-tier GPU sessions do run
in parallel, but no more than two at a time, and the weekly quota of 30 h
applies on top of that.

To add a seed, prefer **Copy & Edit** from a notebook already set to T4 x2
and change only the seed. Copies inherit the accelerator; a fresh CLI push
does not.

A 350-epoch member takes about 1.3 h at this dataset size. Staging costs
about 15 s, since the published dataset arrives pre-extracted under
`/kaggle/input` and the cell only symlinks it, so a restart is cheap. Check
`wallclock_s` in `history.json` before assuming two runs fit together.

In [ ]:
import os, shutil, subprocess, sys, zipfile

REPO = "/kaggle/working/transolver-hypersonic"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/zeteixeira03/transolver-hypersonic.git", REPO],
                   check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "einops"], check=True)

MIN_CASES = 750  # the published dataset carries 753 converged cases

# locate the three ingredients independently: kaggle sometimes auto-extracts
# uploaded archives into a subdirectory, so ledger.db, the case files, and
# su2_cases.zip are not guaranteed to share a directory
INPUT = "/kaggle/input"
assert os.path.isdir(INPUT), "no /kaggle/input at all: attach the dataset"
ledger_path, zip_path, case_dir, case_n = None, None, None, 0
for root, dirs, files in os.walk(INPUT, followlinks=True):
    if "ledger.db" in files and ledger_path is None:
        ledger_path = os.path.join(root, "ledger.db")
    if "su2_cases.zip" in files and zip_path is None:
        zip_path = os.path.join(root, "su2_cases.zip")
    n = sum(1 for f in files if f.startswith("case_") and f.endswith(".npz"))
    if n > case_n:
        case_n, case_dir = n, root
print("ledger:", ledger_path)
print("zip:", zip_path)
print("cases:", case_n, "in", case_dir)
assert ledger_path is not None, f"no ledger.db under {INPUT}; attach the su2 dataset"

if case_n >= MIN_CASES:
    # case files already extracted; make sure the ledger sits next to them
    DATA = case_dir
    if not os.path.isfile(f"{DATA}/ledger.db"):
        DATA = "/kaggle/working/su2_data"
        os.makedirs(DATA, exist_ok=True)
        for f in os.listdir(case_dir):
            if f.startswith("case_") and f.endswith(".npz"):
                dst = f"{DATA}/{f}"
                if not os.path.exists(dst):
                    os.symlink(f"{case_dir}/{f}", dst)
        shutil.copyfile(ledger_path, f"{DATA}/ledger.db")
else:
    assert zip_path is not None, (
        f"only {case_n} case files and no su2_cases.zip under {INPUT}; "
        f"wrong dataset version attached? (expect {MIN_CASES}+ cases)"
    )
    DATA = "/kaggle/working/su2_data"
    if not os.path.isfile(f"{DATA}/ledger.db"):
        os.makedirs(DATA, exist_ok=True)
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(DATA)
        shutil.copyfile(ledger_path, f"{DATA}/ledger.db")

n_cases = len([d for d in os.listdir(DATA) if d.startswith("case_")])
print("DATA:", DATA, "cases:", n_cases)
assert n_cases >= MIN_CASES, (
    f"only {n_cases} case files under {DATA}; wrong dataset version? "
    f"(expect {MIN_CASES}+, including the loop cases)"
)

M = 32   # slice count: the in-distribution optimum from the slice ablation


def run(cmd, log_dir=None):
    # stream child output into the cell; ! magics can't loop with error checks.
    # also tee to the run directory: cell output dies when the notebook is
    # re-pushed, and the log is the only record of warnings and timings
    lines = []
    p = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
        lines.append(line)
    code = p.wait()
    if log_dir:
        os.makedirs(log_dir, exist_ok=True)
        with open(f"{log_dir}/train_log.txt", "w") as f:
            print("$", " ".join(cmd), file=f)
            print(file=f)
            f.writelines(lines)
    if code != 0:
        raise RuntimeError(f"run failed: {' '.join(cmd)}")

In [ ]:
RUN_ENSEMBLE = True         # set False when this copy runs the prior ablation

INIT_SEEDS = [2]            # one seed per notebook copy; see the header
EPOCHS = 350
TAG = "v3"                  # v2 = sweep only; v3 = sweep plus the loop-acquired cases

for s in INIT_SEEDS if RUN_ENSEMBLE else []:
    print(f"===== init_seed={s} =====", flush=True)
    OUT = f"/kaggle/working/run_m{M}_{TAG}_s{s}"
    run([sys.executable, "scripts/train.py",
         "--workdir", DATA,
         "--out", OUT,
         "--slice-num", str(M),
         "--epochs", str(EPOCHS),
         "--val-every", "10", "--seed", "0",
         "--init-seed", str(s)], log_dir=OUT)

## Physics-prior ablation

Prices three priors the surrogate normally carries, one at a time, on a
100-case training split. Priors are meant to earn their place when data is
scarce, so the small split is the design point rather than a shortcut.

The eval tiers are pinned from `configs/prior_splits_100.json`, so every cell
is scored on identical held-out cases at full size and only the train split
is small. Watch the split line: it must read `train: 100, val: 67, test: 65,
test_interp: 54`. Anything else means the pinned file did not resolve against
the attached dataset and the cell is void.

All cells use cosine decay over 500 epochs. That differs from the 350-epoch
constant-LR recipe the ensemble runs use, so these numbers are comparable to
each other but not to the ensemble results.

Five cells across two concurrent sessions: put `base`, `free_p`,
`plain_norm` in one and `qw_direct`, `qw_resid` in the other, then repeat
per seed.

In [ ]:
RUN_PRIOR_ABLATION = False     # set True (and RUN_ENSEMBLE False) for this copy

PRIOR_SEED = 0                 # one seed per round; 0, then 1, then 2
PRIOR_CELLS = ["base", "free_p", "plain_norm"]   # other session takes the q_w pair
PRIOR_EPOCHS = 500
PRIOR_SPLITS = f"{REPO}/configs/prior_splits_100.json"

# baseline carries no flags: hard EoS and log10 on rho/T are the defaults
PRIOR_FLAGS = {
    "base":       [],
    "free_p":     ["--predict-p"],
    "plain_norm": ["--log-targets", "none"],
    "qw_direct":  ["--qw-head", "direct"],
    "qw_resid":   ["--qw-head", "residual"],
}

for cell in PRIOR_CELLS if RUN_PRIOR_ABLATION else []:
    assert os.path.isfile(PRIOR_SPLITS), (
        f"no {PRIOR_SPLITS}; the clone predates the prior ablation, re-clone"
    )
    print(f"===== {cell} seed={PRIOR_SEED} =====", flush=True)
    OUT = f"/kaggle/working/run_{cell}_s{PRIOR_SEED}"
    run([sys.executable, "scripts/train.py",
         "--workdir", DATA,
         "--pinned-splits", PRIOR_SPLITS,
         "--out", OUT,
         "--slice-num", str(M),
         "--epochs", str(PRIOR_EPOCHS),
         "--lr-schedule", "cosine",
         "--val-every", "10", "--seed", "0",
         "--init-seed", str(PRIOR_SEED),
         *PRIOR_FLAGS[cell]], log_dir=OUT)

In [ ]:
import glob, json

for path in sorted(glob.glob("/kaggle/working/run_*/final_eval.json")):
    with open(path) as f:
        final = json.load(f)
    a = final["args"]
    print("=====", path, "=====")
    print("  slice_num:", a["slice_num"], "| init_seed:", a.get("init_seed"),
          "| qw_head:", a.get("qw_head"), "| predict_p:", a.get("predict_p"))
    print("  best epoch:", final.get("best_epoch"), "of", a["epochs"],
          "| best val mean rL2:", final.get("best_val_mean_rL2"))
    print("  env:", final.get("env"))
    print(json.dumps(final["final"], indent=2))
    print("  splits:", final["splits"])